# 1. Plan

Define features as level 1 headings with key activities.
- Environment setup.
- Basic weather data ingestion from URL.
- Parse data into dataframe.

## Environment setup

- Test jupyter kernel is working.
- Ensure libraries are imported.

### Test Kernel is working

In [ ]:
print ("Hello, World!")

### Import relevant libraries

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Ingestion of Weather Data

- Get data via url.
- Paramatise the url to get based on wether type and region.
- Parse into DataFrame.

## 2. Get data from url

### 3. Attempt to call url to download data

In [ ]:
url= "https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/Sunshine/date/East_Anglia.txt"
response = requests.get(url)
print (response.text)

### 3. Parameterise the url

In [ ]:
# This gives a url like this
parameter = "Sunshine"
region = "East_Anglia"
url= f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{parameter}/date/{region}.txt"
response = requests.get(url)
print(response.text)

# 3. Parse into Pandas DataFrame

- Take retrieved data and try parse into a dataframe.

In [ ]:
# Read first few rows of the response text.
weather_data = response.text.splitlines()

for i, line in enumerate(weather_data):
	if i < 7:  # Print first 3 lines
		print(F'Line {i} is {line}')

This indicates the first five rows of data are only **Contextual information** and not part of the dataset proper. This is not needed and will need to be removed for the purposes of data analysis.

The first row of data needed, is the year row for column headings. Then the data following that. Headers are on row 6, and data starts on row 7.

## First five lines of actual data

### Headers

In [ ]:
print("Headers of columns:")
for i, line in enumerate(weather_data):
	if i == 5:
		print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

### First five lines of actual data

In [ ]:
print("First 5 lines of raw file:")
for i, line in enumerate(weather_data):
	if i > 5 and i < 10:
		print(f"Line {i+1}: {repr(line)}")  # repr() shows whitespace characters

## Try reading into Padas Data Frame

### Direct reading URL with pandas

In [ ]:
# Try reading with explicit encoding and error handling
try:
    df = pd.read_csv(url, encoding='utf-8')
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

This shows that the data cannot be read in. Suggesting that pandas is unable to determine the correct delimiter in the CSV file. This is down to the file actually being a .txt file, and being delimited by both tabs and spaces.

### Attempt to handle variable spacing and whitespace

In [ ]:
try:
    # Read with flexible whitespace handling
    df = pd.read_csv(url, 
                     sep='\s+',  # Handle variable spacing
                     skipinitialspace=True,   # Skip leading spaces
                     skip_blank_lines=True)   # Skip empty lines
    
    print("\nDataframe info:")
    print(df.info())
    print("\nFirst 5 rows:")
    print(df.head())
except Exception as e:
    print(f"Error: {str(e)}")

**N.B.** have replaced the paramater definition 'delim_whitespace' keyword with "sep='\s+'" as the former is depreciated, as indicated in execution Futurewarning hints.

This appears to fail as it is expecting fewer columns than are presented in the data table proper. This is likely due to the contextual inforamtion presented at the start of the file.

Try with where contextual data has been stripped out using the skiprows parameter.

In [ ]:
try:
	# Read with flexible whitespace handling
	df = pd.read_csv(url, 
					 sep='\s+',  # Handle variable spacing
					 skipinitialspace=True,	# Skip leading spaces
					 skip_blank_lines=True,	# Skip empty lines
					 skiprows=5)
	
	print("\nDataframe info:")
	print(df.info())
	print("\nFirst 5 rows:")
	print(df.head())
except Exception as e:
	print(f"Error: {str(e)}")

### Show the dataframe

In [ ]:
df

# 1. Process all files using Dynamic URL

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter lables. I can extract these and inject them into the URL string used to request the data.

## Parmeter Lables

|Parameter|Label|
|---------|-----|
|Max temp|Tmax|
|Min temp|Tmin|
|Mean temp|Tmean|
|Sunshine|Sunshine|
|Rainfall|Rainfall|
|Rain days ≥1.0mm|Raindays1mm|
|Days of air frost|AirFrost|

In [ ]:
data_weather = ['Tmax','Tmin','Tmean','Sunshine','Rainfall','Raindays1mm','AirFrost']

## Regions Labels

I can extract each of the Region names from the download page source. This gives me the following table.

### Whole list

|ID|Region|Label|
|--|------|-----|
|1|UK|UK|
|2|England|England|
|3|Wales|Wales|
|4|Scotland|Scotland|
|5|Northern Ireland|Northern_Ireland|
|6|England & Wales|England_and_Wales|
|7|England N|England_N|
|8|Englan S|Englan_S|
|9|Scotland N|Scotland_N|
|10|Scotland E|Scotland_E|
|11|Scotland W|Scotland_W|
|12|England E & NE|England_E_and_NE|
|13|England NW/Wales N|England_NW_and_N_Wales|
|14|Midlands|Midlands|
|15|East Anglia|East_Anglia|
|16|England SW/Wales S|England_SW_and_S_Wales|
|17|England SE/Central S|England_SE_and_Central_S|

### Countries

|ID|Region|Label|
|--|------|-----|
|1|UK|UK|
|2|England|England|
|3|Wales|Wales|
|4|Scotland|Scotland|
|5|Northern Ireland|Northern_Ireland|

### Regions

|ID|Region|Label|
|--|------|-----|
|6|England & Wales|England_and_Wales|
|7|England N|England_N|
|8|England S|England_S|
|9|Scotland N|Scotland_N|
|10|Scotland E|Scotland_E|
|11|Scotland W|Scotland_W|
|12|England E & NE|England_E_and_NE|
|13|England NW/Wales N|England_NW_and_N_Wales|
|14|Midlands|Midlands|
|15|East Anglia|East_Anglia|
|16|England SW/Wales S|England_SW_and_S_Wales|
|17|England SE/Central S|England_SE_and_Central_S|

In [ ]:
data_region = ['UK','England','Wales','Scotland','Northern_Ireland','England_and_Wales','England_N','England_S','Scotland_N','Scotland_E','Scotland_W','England_E_and_NE','England_NW_and_N_Wales','Midlands','East_Anglia','England_SW_and_S_Wales','England_SE_and_Central_S']